# Polymarket crypto-market evaluation

Scores Polymarket **crypto** prediction mids against the **realized CEX price** — the honest test of whether the prediction market is well-calibrated. Reads `pm_book` (mids over time) + `cex_spot` (ground truth) from `data/defi.duckdb`.

Markets like *"Will Bitcoin be above \$68,000 on June 2?"* have a price-threshold truth we already capture, so they're scoreable. **This populates as markets resolve** — until then it reports what's pending.

In [ ]:
import sys, re
from pathlib import Path
import duckdb, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style='whitegrid')
ROOT = Path.cwd()
while not (ROOT/'data').exists() and ROOT != ROOT.parent: ROOT = ROOT.parent
con = duckdb.connect(str(ROOT/'data'/'defi.duckdb'), read_only=True)
CEX_ASSETS = {r[0] for r in con.execute("select distinct asset from cex_spot").fetchall()}
print('CEX ground-truth assets:', CEX_ASSETS)

## Parse tracked markets

Extract (asset, threshold, direction, resolution date) from each question. Markets we can't parse, or whose asset we don't price, are skipped.

In [ ]:
ASSET_MAP = {'bitcoin':'BTC','btc':'BTC','ethereum':'ETH','ether':'ETH','solana':'SOL',
             'dogecoin':'DOGE','doge':'DOGE','xrp':'XRP','ripple':'XRP'}
def parse_q(q):
    ql = q.lower()
    asset = next((v for k,v in ASSET_MAP.items() if re.search(r'\b'+k+r'\b', ql)), None)
    m = re.search(r'\$\s*([\d,]+(?:\.\d+)?)\s*([kKmM]?)', q)
    thr = None
    if m:
        thr = float(m.group(1).replace(',',''))
        thr *= {'k':1e3,'m':1e6}.get(m.group(2).lower(), 1)
    direction = 'below' if ('below' in ql or 'under' in ql) else 'above'
    dm = re.search(r'(?:on|by)\s+([A-Za-z]+\.?\s+\d{1,2}(?:,?\s+\d{4})?)', q)
    when = None
    if dm:
        s = dm.group(1)
        if not re.search(r'\d{4}', s): s += ' 2026'
        try: when = pd.to_datetime(s)
        except Exception: when = None
    return asset, thr, direction, when

mkts = con.execute('SELECT DISTINCT condition_id, question FROM pm_book').df()
parsed = []
for _, r in mkts.iterrows():
    a, thr, d, when = parse_q(r.question)
    parsed.append({'condition_id': r.condition_id, 'question': r.question[:60],
                   'asset': a, 'threshold': thr, 'direction': d, 'resolves': when,
                   'scoreable': bool(a in CEX_ASSETS and thr and when is not None)})
pf = pd.DataFrame(parsed)
print(f'{len(pf)} markets | {pf.scoreable.sum()} scoreable (asset priced + threshold + date)')
pf

## A market's mid over time

The prediction evolving toward resolution (fills as the 30-min cron runs).

In [ ]:
hist = con.execute('''
  SELECT captured_at, question, outcome, mid FROM pm_book
  WHERE outcome = 'Yes' ORDER BY captured_at''').df()
if hist['captured_at'].nunique() >= 2 and len(hist):
    top_q = hist.question.value_counts().index[0]
    d = hist[hist.question == top_q]
    plt.figure(figsize=(11,4)); plt.plot(d.captured_at, d.mid, marker='o')
    plt.title(f'YES mid over time: {top_q[:70]}'); plt.ylabel('mid (implied P)'); plt.ylim(0,1); plt.show()
else:
    print(f"Only {hist['captured_at'].nunique()} snapshot(s) so far — mid time series fills as the 30-min cron runs.")

## Resolve & score

For each scoreable market whose resolution time has passed and for which we've captured a realized CEX price: outcome = did price cross the threshold; prediction = the last YES mid before resolution. Accumulates (pred, outcome).

In [ ]:
def realized_price(asset, when):
    row = con.execute('''SELECT last FROM cex_spot WHERE asset=? AND venue='coinbase'
        AND captured_at >= ? ORDER BY captured_at LIMIT 1''', [asset, when]).fetchone()
    return row[0] if row else None
def last_yes_mid(cid, when):
    row = con.execute('''SELECT mid FROM pm_book WHERE condition_id=? AND outcome='Yes'
        AND captured_at <= ? ORDER BY captured_at DESC LIMIT 1''', [cid, when]).fetchone()
    return row[0] if row else None

scored, pending = [], 0
for _, r in pf[pf.scoreable].iterrows():
    px = realized_price(r.asset, r.resolves)
    pred = last_yes_mid(r.condition_id, r.resolves)
    if px is None or pred is None:
        pending += 1; continue
    outcome = int(px >= r.threshold) if r.direction=='above' else int(px < r.threshold)
    scored.append({'question': r.question, 'asset': r.asset, 'thr': r.threshold,
                   'realized': px, 'pred_yes': pred, 'outcome': outcome})
sf = pd.DataFrame(scored)
print(f'{len(sf)} resolved & scored | {pending} scoreable but pending (no realized price / mid yet)')
sf

## Calibration & Brier of the prediction market

Brier = mean((pred − outcome)²); lower is better. The reliability curve shows whether a 70%-mid actually wins ~70% of the time.

In [ ]:
if len(sf) >= 10:
    brier = float(np.mean((sf.pred_yes - sf.outcome)**2))
    print(f'Brier {brier:.4f} over {len(sf)} resolved markets (base rate {sf.outcome.mean():.3f})')
    from sklearn.calibration import calibration_curve
    fp, mp = calibration_curve(sf.outcome, sf.pred_yes, n_bins=min(10, len(sf)//3), strategy='quantile')
    plt.figure(figsize=(6,6)); plt.plot([0,1],[0,1],'k--', label='perfect')
    plt.plot(mp, fp, 'o-', label='Polymarket mid')
    plt.xlabel('mid (implied P)'); plt.ylabel('realized frequency')
    plt.title('Polymarket crypto calibration'); plt.legend(); plt.show()
else:
    print(f'{len(sf)} resolved markets — need >=10 for a calibration read.')
    print('This is the cell to watch: as daily BTC/ETH threshold markets settle, it fills in.')